In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from scipy import stats as st
import os
import csv
from collections import Counter
from difflib import SequenceMatcher

In [2]:
def string_similarity_match(str1, str2, threshold=90):
    # Calculate similarity ratio using SequenceMatcher
    ratio = SequenceMatcher(None, str1.lower(), str2.lower()).ratio()
    similarity_percentage = ratio * 100 
    
    return similarity_percentage >= threshold

def get_best_match_index(target_string, candidate_list, threshold=90):
    if not candidate_list:
        return None
    
    best_similarity = 0.0
    best_index = None
    
    for i, candidate in enumerate(candidate_list):
        ratio = SequenceMatcher(None, str(target_string).lower(), str(candidate).lower()).ratio()
        similarity = ratio * 100
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_index = i
    
    # Return index only if it meets threshold
    if best_similarity >= threshold:
        print('best sim:', best_similarity)
    return best_index if best_similarity >= threshold else None

In [3]:
def transform_sp24_dict(original_dict):
    
    transformed_dict = {'message': [], 'label': []}
    
    # Process ham messages (label = 0)
    for message in original_dict['ham']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('ham')
    
    # Process spam messages (label = 1)
    for message in original_dict['spam']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('spam')
    
    return pd.DataFrame(transformed_dict)

In [4]:
def read_file(file_path):
    with open(file_path, 'r') as file:
        content = file.readlines()
    return content

In [5]:
sp24_dataset = pd.read_csv('../Dataset/sp24-smishing/phishing_messages.csv', encoding='unicode_escape')
# sp24_dataset = pd.read_csv('../Dataset/sp24-smishing/phishing_campaigns.csv', encoding='unicode_escape')
sp24_dataset = sp24_dataset.dropna()
sp24_dataset

,messageID,objectID,destination number,message,X,time,error in time
0,0,627001980f51cabb09ccf227,447883196637,"Join today and enjoy 400% casino bonus, START ...",VIP Club,1.651511e+09,7200.0
1,1,627001ca4bbc0d101565a3c8,436703061433,Einzigartiges Angebot- 200% bis zu 100 EUR +50...,Top 4You,1.651530e+09,120.0
2,2,62734e2618226a2f38733d2c,447591156667,An up to Â£3 FREEBIE has been dropped onto you...,+447903576999,1.651455e+09,0.0
3,3,627001e1438a83d5e42455f8,17312148616,We just missed you today! Schedule a redeliver...,Google,1.651490e+09,0.0
4,4,62715d86a36b4bc273623c5b,18458695131,"In order to avoid any loss to you, please corr...",Telegram,1.651462e+09,0.0
...,...,...,...,...,...,...,...
68024,68024,64b0f2c2b0002d28aba130c3,12542145739,New RHRY L0gin ã www.bitoii.com ã Username...,+16289992825,1.681308e+09,5356800.0
68025,68025,64b0f2ccb0002d28aba136c1,12542145739,New RHRY L0gin ã www.bitoii.com ã Username...,+16289992825,1.681308e+09,5356800.0
68026,68026,64b0f2d8b0002d28aba13e9c,16463614061,sms180217@2-tdbanking.sms.com / Zelle Detected...,+6245,1.683986e+09,5356800.0
68027,68027,64b0f2d8b0002d28aba13ec9,16463614061,sms370017@2-tdbanking.sms.com / Zelle Detected...,+6245,1.683986e+09,5356800.0


In [6]:
# sp24_dataset = transform_sp24_dict(sp24_dataset)
# sp24_dataset.head()

In [7]:
# Counter(pd.read_csv('../Dataset/URL Data/'+'sp24 Dataset_'+'.csv')['URL'].to_list())

In [8]:
sp24_dataset['Extracted URL'] = pd.read_csv('../Dataset/URL Data/'+'sp24 Dataset_'+'.csv')['URL']
sp24_dataset['Message Len'] = [len(str(i)) for i in sp24_dataset['message']]
sp24_dataset.head()

,messageID,objectID,destination number,message,X,time,error in time,Extracted URL,Message Len
0,0,627001980f51cabb09ccf227,447883196637,"Join today and enjoy 400% casino bonus, START ...",VIP Club,1.651511e+09,7200.0,https://tx.vc/r/2OJ4A/1zBJ2x/7SSQfsp,112
1,1,627001ca4bbc0d101565a3c8,436703061433,Einzigartiges Angebot- 200% bis zu 100 EUR +50...,Top 4You,1.651530e+09,120.0,https://tx.vc/r/2OJ7W/1zBJrA/7GGdxz3,131
2,2,62734e2618226a2f38733d2c,447591156667,An up to Â£3 FREEBIE has been dropped onto you...,+447903576999,1.651455e+09,0.0,http://pktwn.uk/184ec,145
3,3,627001e1438a83d5e42455f8,17312148616,We just missed you today! Schedule a redeliver...,Google,1.651490e+09,0.0,NaN,115
4,4,62715d86a36b4bc273623c5b,18458695131,"In order to avoid any loss to you, please corr...",Telegram,1.651462e+09,0.0,https://www.uspocl.com,110


In [9]:
sp24_website_analysis_data = pd.read_csv('../Dataset/URL Data/'+'SMSGateway Websites Analysis'+'.csv')
# sp24_website_analysis_data = sp24_website_analysis_data.drop(columns=['ham', 'spam'])
sp24_website_analysis_data.head()

,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,https://tx.vc/r/2OJ4A/1zBJ2x/7SSQfsp,tx.vc,0,0,404,0
1,https://tx.vc/o/804d5,tx.vc,0,0,404,0
2,https://tx.vc/r/2OJ7W/1zBJrA/7GGdxz3,tx.vc,0,0,404,0
3,https://tx.vc/o/804d5,tx.vc,0,0,404,0
4,http://pktwn.uk/184ec,pktwn.uk,0,0,-1,0


In [10]:
string_similarity_match('http://wiseschool.com','wiseschool.com')

False

In [11]:
sp24_website_analysis_data.iloc[0][0]

C:\Users\mmia43\AppData\Local\Temp\ipykernel_23984\3578951940.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  sp24_website_analysis_data.iloc[0][0]


'https://tx.vc/r/2OJ4A/1zBJ2x/7SSQfsp'

In [12]:
# for row in sp24_website_analysis_data.itertuples(index=False):
#     print(row[0])

In [13]:
import tldextract

def FQDN(Url):
    
    if type(Url) !=str:
        return ''

    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [14]:
extracted_urls = sp24_dataset['Extracted URL'].values
extracted_urls_fqdn = [FQDN(i) for i in extracted_urls]

# Create a dictionary for O(1) lookup instead of O(n) loop
fqdn_to_index = {fqdn: idx for idx, fqdn in enumerate(sp24_website_analysis_data['FQDN'])}
website_data = sp24_website_analysis_data.iloc[:, 1:6].values

# Pre-allocate lists for better performance
n_urls = len(extracted_urls)
fqdn = extracted_urls_fqdn
website_size = [''] * n_urls
text_content_len = [''] * n_urls
status_code = [''] * n_urls
parked = [''] * n_urls

# Single optimized loop with dictionary lookup
for i, url in enumerate(extracted_urls):
    if url and extracted_urls_fqdn[i]:  # Check both url and fqdn exist
        matched_idx = fqdn_to_index.get(extracted_urls_fqdn[i])  # O(1) lookup
        if matched_idx is not None:
            row_data = website_data[matched_idx]
            website_size[i] = row_data[1]
            text_content_len[i] = row_data[2]
            status_code[i] = row_data[3]
            parked[i] = row_data[4]

In [15]:
sp24_dataset['FQDN'] = fqdn
sp24_dataset['Website Size in KB'] = website_size
sp24_dataset['Website Textual Content Length'] = text_content_len
sp24_dataset['Status Code'] = status_code
sp24_dataset['Parked'] = parked

In [16]:
sp24_dataset = sp24_dataset.replace('', np.nan)
sp24_dataset.head()

C:\Users\mmia43\AppData\Local\Temp\ipykernel_23984\781432750.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sp24_dataset = sp24_dataset.replace('', np.nan)


,messageID,objectID,destination number,message,X,time,error in time,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,0,627001980f51cabb09ccf227,447883196637,"Join today and enjoy 400% casino bonus, START ...",VIP Club,1.651511e+09,7200.0,https://tx.vc/r/2OJ4A/1zBJ2x/7SSQfsp,112,tx.vc,0.0,0.0,404.0,0.0
1,1,627001ca4bbc0d101565a3c8,436703061433,Einzigartiges Angebot- 200% bis zu 100 EUR +50...,Top 4You,1.651530e+09,120.0,https://tx.vc/r/2OJ7W/1zBJrA/7GGdxz3,131,tx.vc,0.0,0.0,404.0,0.0
2,2,62734e2618226a2f38733d2c,447591156667,An up to Â£3 FREEBIE has been dropped onto you...,+447903576999,1.651455e+09,0.0,http://pktwn.uk/184ec,145,pktwn.uk,0.0,0.0,-1.0,0.0
3,3,627001e1438a83d5e42455f8,17312148616,We just missed you today! Schedule a redeliver...,Google,1.651490e+09,0.0,NaN,115,NaN,NaN,NaN,NaN,NaN
4,4,62715d86a36b4bc273623c5b,18458695131,"In order to avoid any loss to you, please corr...",Telegram,1.651462e+09,0.0,https://www.uspocl.com,110,www.uspocl.com,0.0,0.0,-1.0,0.0


In [17]:
# Counter(sp24_dataset['FQDN'].to_list())
sp24_dataset[(sp24_dataset['Extracted URL'].notna()) & (sp24_dataset['FQDN'].isna())]

,messageID,objectID,destination number,message,X,time,error in time,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked


In [18]:
# Counter(sp24_dataset['FQDN'].to_list())
sp24_dataset[(sp24_dataset['Extracted URL'].notna()) & (sp24_dataset['FQDN'].notna())]

,messageID,objectID,destination number,message,X,time,error in time,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,0,627001980f51cabb09ccf227,447883196637,"Join today and enjoy 400% casino bonus, START ...",VIP Club,1.651511e+09,7200.0,https://tx.vc/r/2OJ4A/1zBJ2x/7SSQfsp,112,tx.vc,0.0,0.0,404.0,0.0
1,1,627001ca4bbc0d101565a3c8,436703061433,Einzigartiges Angebot- 200% bis zu 100 EUR +50...,Top 4You,1.651530e+09,120.0,https://tx.vc/r/2OJ7W/1zBJrA/7GGdxz3,131,tx.vc,0.0,0.0,404.0,0.0
2,2,62734e2618226a2f38733d2c,447591156667,An up to Â£3 FREEBIE has been dropped onto you...,+447903576999,1.651455e+09,0.0,http://pktwn.uk/184ec,145,pktwn.uk,0.0,0.0,-1.0,0.0
4,4,62715d86a36b4bc273623c5b,18458695131,"In order to avoid any loss to you, please corr...",Telegram,1.651462e+09,0.0,https://www.uspocl.com,110,www.uspocl.com,0.0,0.0,-1.0,0.0
7,7,62701fb12e5a9b0560aac0e8,61481272690,An unauthorised transaction was perfor...,Instagram,1.651514e+09,0.0,https://permanenttsb-assist.com,178,permanenttsb-assist.com,0.0,0.0,-1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68024,68024,64b0f2c2b0002d28aba130c3,12542145739,New RHRY L0gin ã www.bitoii.com ã Username...,+16289992825,1.681308e+09,5356800.0,www.bitoii.com,99,www.bitoii.com,0.0,0.0,200.0,0.0
68025,68025,64b0f2ccb0002d28aba136c1,12542145739,New RHRY L0gin ã www.bitoii.com ã Username...,+16289992825,1.681308e+09,5356800.0,www.bitoii.com,99,www.bitoii.com,0.0,0.0,200.0,0.0
68026,68026,64b0f2d8b0002d28aba13e9c,16463614061,sms180217@2-tdbanking.sms.com / Zelle Detected...,+6245,1.683986e+09,5356800.0,https://acortartu.link/vu4ne,160,acortartu.link,0.0,0.0,-1.0,0.0
68027,68027,64b0f2d8b0002d28aba13ec9,16463614061,sms370017@2-tdbanking.sms.com / Zelle Detected...,+6245,1.683986e+09,5356800.0,https://acortartu.link/vu4ne,160,acortartu.link,0.0,0.0,-1.0,0.0


In [19]:
print(len(sp24_dataset))

68029


In [20]:
#messages with URL
print(len(sp24_dataset[(sp24_dataset['Extracted URL'].notna())]), len(sp24_dataset[(sp24_dataset['Extracted URL'].notna())])/len(sp24_dataset))

41676 0.6126210880653838


In [21]:
# #smish messages with URL
# len(sp24_dataset[(sp24_dataset['Extracted URL'].notna()) & (sp24_dataset['class']==1)])

In [22]:
# #smish messages with URL
# len(sp24_dataset[(sp24_dataset['Extracted URL'].notna()) & (sp24_dataset['class']==0)])

In [23]:
#unique FQDN
len(set(sp24_dataset[(sp24_dataset['FQDN'].notna())]['FQDN']))

842

In [24]:
only_unique_live_websites_data = sp24_dataset.drop_duplicates(subset=['FQDN'])
#live websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200)]))

107


In [25]:
#parked websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1)]))

29


In [26]:
sp24_dataset.to_csv('../Dataset/Refined_SMS_Gateway_Dataset.csv', index=None)